# Step 1

Run the following 6 "cells" by selecting them and pressing "Shift+Enter".

In [ ]:
# Cell 1

import numpy as np
from scipy.optimize import fsolve, brentq
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display, Math
#hbar = 0.6582 # in [10^-3 eV s]
hbar = 1 # dummy hbar, not used

In [ ]:
# Cell 2

def k(E,m,hbar):
    #return np.sqrt(2*m*E)/hbar
    return 0.22664*np.sqrt(m*E) # in units of [nm^-1]
    
def q(E,V0,m,hbar):
    #return np.sqrt(2*m*(V0-E))/hbar
    return 0.22664*np.sqrt(m*(V0-E)) # in units of [nm^-1]

class even:
    def __init__(self, E, n, V0, m, L):
        self.n = n+1
        self.L = L
        self.lower = -L/2.0
        self.upper = L/2.0    
        self.k = k(E,m,hbar)
        self.q = q(E,V0,m,hbar)
        self.normalize()
    
    def normalize(self):
        self.D = 1.0/np.sqrt(self.L/2.0 + 1.0/self.q)
        self.A = np.exp(self.q*self.L/2.0)*np.cos(self.k*self.L/2.0)*self.D
    
    def value(self,x):
        if x < self.lower:
            return self.A*np.exp(self.q*x)
        elif self.lower <= x and x <= self.upper:
            return self.D*np.cos(self.k*x)
        elif self.upper < x:
            return self.A*np.exp(-self.q*x)
    
    def print_normalization(self):
        print(f"A = {self.A}")
        print(f"D = {self.D}")
    
    
    
class odd:
    def __init__(self, E, n, V0, m, L):
        self.n = n+1
        self.L = L
        self.lower = -L/2.0
        self.upper = L/2.0    
        self.k = k(E,m,hbar)
        self.q = q(E,V0,m,hbar)
        self.normalize()
    
    def normalize(self):
        self.C = 1.0/np.sqrt(self.L/2.0 + 1.0/self.q)
        self.A = -np.exp(self.q*self.L/2.0)*np.sin(self.k*self.L/2.0)*self.C

    
    def value(self,x):
        if x < self.lower:
            return self.A*np.exp(self.q*x)
        elif self.lower <= x and x <= self.upper:
            return self.C*np.sin(self.k*x)
        elif self.upper < x:
            return -self.A*np.exp(-self.q*x)

    def print_normalization(self):
        print(f"A = {self.A}")
        print(f"C = {self.C}")


In [ ]:
# Cell 3

def plot_eigenstate(phi,prob=False):
    fig, axs = plt.subplots(1,2,layout='constrained',figsize=(9,4))

    inf_factor = 2
    xmin = -inf_factor*L/2.0
    xmax = inf_factor*L/2.0
    x_array = np.linspace(xmin,xmax,200)
    
    for ax in axs:
        ax.set_xlim(xmin,xmax)
        ax.set_xlabel('x [nm]')
        ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.2)) 
        ax.axvline(-phi.L/2.0,color='black',ls='--',label=r"$V_0$")
        ax.axvline(phi.L/2.0,color='black',ls='--')
        ax.axhline(0,color='black')
    
    
    # Wavefunction
    y = np.array( [ phi.value(x) for x in x_array ], dtype=np.float32)
    axs[0].plot(x_array,y,label=rf"$\phi_{{E_{phi.n}}}$")
    axs[0].legend()
    axs[0].set_ylabel(r"$Re(\phi)$")
    
    # Probability
    y = np.array( [ phi.value(x)**2 for x in x_array ], dtype=np.float32)
    axs[1].plot(x_array,y,label=rf"$\phi_{{E_{phi.n}}}$")
    axs[1].set_ylabel(r"$|\phi|^2$")
    indices = np.where(x_array < phi.lower)
    axs[1].fill_between(x_array[indices],y[indices],color="#c8c8c8")
    indices = np.where(x_array > phi.upper)
    axs[1].fill_between(x_array[indices],y[indices],color="#c8c8c8")
    
    fig.suptitle(f'Eigenstate for n={phi.n}', fontsize=16)
    
    plt.show()

In [ ]:
# Cell 4

def circle(z,z0):
    return np.sqrt(z0**2 - z**2)

def func1(z):
    return z*np.sin(z)/np.cos(z)

def func2(z):
    return -z*np.cos(z)/np.sin(z)

def equation_even(z,z0):
    return func1(z) - circle(z,z0)

def equation_odd(z,z0):
    return func2(z) - circle(z,z0)

def z0_func(V0,m,L):
    #return np.sqrt(2*m*V0*L**2/(4*hbar**2))    
    return 0.113318*np.sqrt(m*V0)*L # dimensionless
    
def energy(z,m,L):
    #return 4*z**2*hbar**2/(2*m*L**2)
    return 77.8757*z**2/(m*L**2) # in units of [10^-3 eV]

In [ ]:
# Cell 5

def plot_energy_roots(V0,m,L):
    z0 = z0_func(V0,m,L)

    z_array = np.linspace(0,z0,100)
    circl = [ circle(z,z0) for z in z_array ]
    
    fig, ax = plt.subplots(layout='constrained')

    ax.set_aspect('equal')
    ax.set_xlim(0,1.2*z0)
    ax.set_ylim(0,1.2*z0)
    ax.set_xlabel('z')
    ax.set_ylabel('f(z)')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.5))
    
    # Plot circle
    circ, = ax.plot(z_array,circl,label=r"$\sqrt{z^2 - z_0^2}$")

    # Plot solutions with tangent
    step_size = np.pi/50.0
    n = 0
    while n*np.pi < z0:
        lower = n*np.pi
        upper = min((n+1/2)*np.pi,z0)
        N = int(np.ceil((upper-lower)/step_size))
        xs = [0]*N
        ys = [0]*N
        for i in range(0,N):
            x = lower + i*step_size
            xs[i] = x
            ys[i] = func1(x)
        even, = ax.plot(xs,ys,color='red',label=r"$z \, \mathrm{tan} (z)$")
        n = n + 1
    
    # Plot solutions with cotangent
    n = 0
    while n*np.pi < z0:
        lower = (n+1/2)*np.pi
        upper = min((n+1)*np.pi,z0)
        N = int(np.ceil((upper-lower)/step_size))
        xs = [0]*N
        ys = [0]*N
        for i in range(0,N):
            x = lower + i*step_size
            xs[i] = x
            ys[i] = func2(x)
        odd, = ax.plot(xs,ys,color='orange',label="$-z \, \mathrm{cot} (z)$")
        n = n + 1
    
    # Plot roots (Energy)
    x_roots,y_roots = get_roots(V0,m,L)    
    ax.scatter(x_roots,y_roots)
    
    ax.legend(handles=[circ,even,odd])
    
    plt.show()

In [ ]:
# Cell 6

def get_roots(V0,m,L):
    z0 = z0_func(V0,m,L)

    # Solve in the range [nπ,(n+1)π]
    roots_even = []
    n = 0
    while n*np.pi < z0:
        lower = n*np.pi
        upper = min((n+1/2)*np.pi,z0)
        root = brentq(equation_even,lower,upper,args=(z0))
        roots_even.append(root)
        n = n + 1
    
    roots_odd  = []
    n = 0
    while (n+1/2)*np.pi < z0:
        lower = (n+1/2)*np.pi
        upper = min((n+1)*np.pi,z0)
        #print(n,lower,upper,equation_odd(lower,z0),equation_odd(upper,z0))
        root = brentq(equation_odd,lower,upper,args=(z0))
        roots_odd.append(root)
        n = n + 1
    
    x_roots = np.sort(np.concatenate((roots_even,roots_odd)))
    y_roots = [ circle(z,z0) for z in x_roots ]

    return x_roots,y_roots

# Step 2

The following cell calculates the allowed energies in the finite potential well based on 3 parameters: the **width** of the well, **L**, the **finite value of the potential, V<sub>0</sub>**, and the **mass** of the particle inside the well, **m**. These are the ''free'' parameters of the problem, meaning that you can change them at will and examine different cases. <u>Anytime you want to change one of these parameters, you will have to run all the cells below this one</u>.
<br><br>

Note that the units of energy used in the book are: 
$$\frac{2 \hbar}{m L^2},$$
<br>
which means that they depend on the mass of the particle and the size of the well. This is a convenient unit to examine the behaviour of many such quantum systems despite big differences in their properties, e.g. the electron has a mass of 0.511 MeV, while the proton 938.3 MeV and the neutron 939.6 MeV.
Here, we make a specific choice of units: nano-meters (nm$=10^{-9}$m) for length, mili-eV (meV=$10^{-3}$eV), for the energy and the potential, and Mega-eV (MeV$=10^6$eV) for the mass. In practice, because we make these specific choices, we need to pay attention to the ranges of the quantities that we want to explore for different particles, e.g. $V_0$, mass, and width.
<br><br>

In order to reproduce the values in equation 5.89 of the book, you need to set $L=1, V_0 = 2803$, and $m=1$. If you do this, then you will find the correct solutions of the transcedental equations ($z$). To find the correct values for the energy, you will need to a) explicitly calculate the arbitrary energy unit $2\hbar^2/mL^2$, b) convert it to milli-eV, and c) divide the energy values in the cell below with the result.


In [ ]:
L = 10      # width of the potential, in [nano-meters, fm = 10^(-9) m]
V0 = 100  # (finite) height of the walls of the potential, in [mili-eV, meV = 10^(-3) eV]
m = 0.511      # mass of the particle inside the potential well, in [Mega-eV, MeV = 10^6 eV]


# You don't need to change anything else in this cell below this line
plot_energy_roots(V0,m,L)

z_roots,dum = get_roots(V0,m,L)
N_energies = len(z_roots)
energies = [0]*N_energies

print(f"Found {N_energies} allowed energy values [in 10^-3 eV]:")
for i in range(0,N_energies):
    energies[i] = energy(z_roots[i],m,L)
    print(f"Energy {i+1}: {energies[i]:.2f} (z={z_roots[i]:.2f})")

# Step 3

Run the following cell that computes the energy eigenfunctions based on the L, V<sub>0</sub>, and m that were provided above and the corresponding energies. You shouldn't modify anything in this cell.

In [ ]:
phis = [0]*N_energies

for i in range(0,N_energies):
    if i % 2 == 0:
        # even
        phis[i] = even(energies[i],i,V0,m,L)
    else:
        # odd
        phis[i] = odd(energies[i],i,V0,m,L)
    

# Step 4

Select an eigenstate to plot by selecting a value for the index (note that it starts from 0 and has to be less than the number of allowed energies computed above). Note that the cell below reports the value of the factor **q** for the selected eigenstate, as well as the normalization factors **A**, and **C** or **D** depending on whether the eigenstates is even or odd.

In [ ]:
index = 4 # Note that index=n-1, so it starts from 0, and for index=0 we get n=1, etc

# You don't need to change anything else in this cell below this line
if index < 0:
    print("Attention: 'index' must be a positive integer or zero!")
elif not isinstance(index,int):
    print("Attention: 'index' must be an integer number!")
elif index >= N_energies:
    print(f"Attention: There are only {N_energies} levels in this system!")
    print(f"'index' must be in the range (0,{N_energies-1}).")
else:
    target_state = phis[index] 
    plot_eigenstate(target_state)
    print("The normalization coefficients are: ")
    target_state.print_normalization()
    print(f"The factor 'q' is: {target_state.q:.5f} [nm^-1]")
    print(f"The factor 'k' is: {target_state.k:.5f} [nm^-1]")